# Diagnostics, figures and training records

This notebook trains a **tiny** model for a few updates on a small corpus with known truth, decomposes one held-out world, scores the draws against that truth and writes a report. It runs on a CPU in well under a minute, so its numbers demonstrate the API rather than recovery or calibration. Install `pymc-mktfm[diagnostics]`. With the `generation` extra the corpus follows the profile-A recipe at a reduced size; otherwise the notebook builds a small synthetic corpus. No tracking server, network or Hugging Face access is needed.

Every `.plot` method returns a matplotlib `Figure`, which the notebook displays as the cell result; elsewhere, save it with `figure.savefig(path)`. Result, diagnostics and comparison figures state the head, window, mode, units and draw count they show; run evaluation figures state units, scope, legs, draws and head labels; record and corpus figures name the run.

In [ ]:
import importlib.util
from pathlib import Path
from tempfile import TemporaryDirectory

import numpy as np

from pymc_mktfm.diagnostics import TrainingRun
from pymc_mktfm.inference import decompose
from pymc_mktfm.model import ModelConfig
from pymc_mktfm.model.graph import EdgeSlotLayout
from pymc_mktfm.training import TrackingConfig, TrainingConfig, train

workspace = TemporaryDirectory(prefix="mktfm-diagnostics-")
root = Path(workspace.name)

## A corpus with known truth

`synthetic_corpus` builds a schema-4 corpus like the test suite's: the outcome is the exact sum of the components plus noise, the first channel is always active and the last one is padding. A channel or control contributes only through its direct `cy` or `zb` edge, so graph truth decides which effects are live and which are dead.

In [ ]:
def synthetic_corpus(*, n_cells=8, draws_per_cell=2, n_weeks=16, k=4, m=3, j=1, seed=0):
    """A schema-4 corpus; its last two cells are held out and contexts are 8 or 10 weeks."""
    rng = np.random.default_rng(seed)
    n = n_cells * draws_per_cell
    cell_id = np.repeat(np.arange(n_cells, dtype=np.int32), draws_per_cell)
    lengths = np.asarray((8, 10))[cell_id % 2]

    def active(width, padded=False):
        mask = rng.random((n_cells, width)) < 0.7
        mask[:, 0] = True
        if padded:
            mask[:, -1] = False
        return mask[cell_id].astype(np.uint8)

    active_c, active_m, active_j = active(k, padded=True), active(m), active(j)
    layout = EdgeSlotLayout(k, m, j)
    allowed = layout.active_slot_mask(active_c, active_m, active_j) > 0
    graph = ((rng.random((n_cells, layout.n_slots)) < 0.5)[cell_id] & allowed).astype(np.uint8)
    live_c = graph[:, None, layout.slices["cy"]]
    live_m = graph[:, None, layout.slices["zb"]]
    spend = np.exp(rng.normal(8.0, 0.5, (n, n_weeks, k))) * active_c[:, None]
    baseline = 500.0 + rng.normal(0.0, 5.0, (n, n_weeks))
    direct = np.abs(rng.normal(20.0, 5.0, (n, n_weeks, k))) * active_c[:, None] * live_c
    controls = rng.normal(0.0, 1.0, (n, n_weeks, m)) * active_m[:, None]
    covariate = rng.normal(0.0, 3.0, (n, n_weeks, m)) * active_m[:, None] * live_m
    latent = rng.normal(0.0, 2.0, (n, n_weeks, j)) * active_j[:, None]
    indirect = rng.normal(0.0, 1.0, (n, n_weeks, 3))
    noise = rng.normal(0.0, 1.0, (n, n_weeks))
    parts = (baseline, direct.sum(-1), covariate.sum(-1), latent.sum(-1), indirect.sum(-1), noise)
    arrays = {
        "treatment_raw": spend,
        "covariates": controls,
        "outcome_raw": sum(parts),
        "outcome_noise": noise,
        "baseline_intrinsic": baseline,
        "treatment_contribution_raw": direct,
        "covariate_contribution": covariate,
        "latent_unobserved_contribution": latent,
        "indirect_effects_by_source": indirect,
    }
    return {
        "diagnostics": {
            "schema_version": 4,
            "edge_types": ["cy", "dc", "dz", "dy", "zy", "zc", "cc", "zz"],
        },
        "cell_id": cell_id,
        "is_val": (cell_id >= n_cells - 2).astype(np.uint8),
        "support_mask": (np.arange(n_weeks)[None, :] < lengths[:, None]).astype(np.uint8),
        "treatment_active_mask": active_c,
        "covariate_active_mask": active_m,
        "latent_active_mask": active_j,
        "g": graph,
        **{name: values.astype(np.float32) for name, values in arrays.items()},
    }


if importlib.util.find_spec("pymc_generator") is not None:
    from pymc_mktfm.data.profiles import PROFILE_A
    from pymc_mktfm.data.recipes import SHARD_PLANS, generate_corpus, load_corpora, profile_prior

    prior = profile_prior(
        PROFILE_A,
        n_cells=6,
        draws_per_cell=2,
        seed=SHARD_PLANS["A"].seed(0),
        n_time_steps=16,
        l_max=4,
        carryover_burn_in=0,
        query_frac=0.5,
        p_long_horizon=0.0,
    )
    generate_corpus(prior, root / "corpus.npz")
    corpus = load_corpora([root / "corpus.npz"])[0].corpus
    source = "profile-A recipe, 12 worlds of 16 weeks"
else:
    corpus = synthetic_corpus()
    source = "synthetic corpus"
source, corpus["treatment_raw"].shape

## Train a tiny model and read back its records

`train` writes `run.json`, `metrics.jsonl` and `evaluations.jsonl` into its run directory. `TrainingRun.load` reads only those records, never the model, so it also follows a run that is still writing; `trained.records()` returns the same object.

In [ ]:
config = ModelConfig(
    d_model=8,
    n_layers=1,
    n_heads=2,
    d_ff=16,
    flow_hidden=16,
    n_controls=corpus["covariates"].shape[2],
    n_demand=corpus["latent_active_mask"].shape[1],
)
trained = train(
    [corpus],
    out_dir=root / "run",
    model_config=config,
    training=TrainingConfig(
        steps=4,
        batch_size=2,
        warmup_steps=2,
        eval_every=2,
        eval_worlds=2,
        eval_samples=4,
        log_every=1,
        milestone_steps=(),
        seed=5,
    ),
    tracking=TrackingConfig(enabled=False),
    device="cpu",
)
run = TrainingRun.load(root / "run")
run

In [ ]:
run.plot.losses()

In [ ]:
run.plot.learning_rate()

In [ ]:
run.plot.gradients()

In [ ]:
run.plot.evaluation("decomposition_energy_score")

`heads` overlays every head that records a metric: the joint and conditional heads per window, and for edge scores (pass the edge family as the window) the structure head and the joint router's edge marginals.

In [ ]:
run.plot.heads("decomposition_energy_score")

In [ ]:
run.plot.heads("auroc", window="cy")

In [ ]:
run.plot.corpus()

In [ ]:
# Report-only quantities per evaluation event: held-out minus seen gaps of every
# recorded key, and mean AUROC/ECE over edge families. The last event's mean scores:
derived = run.derived_metrics()[-1]
{key: value for key, value in derived.items() if key.endswith(("/mean_auroc", "/mean_ece"))}

## Decompose one held-out world

The model sees the world's spend, controls, context outcome and activity masks only. Its truth stays in the corpus until the diagnostics below.

In [ ]:
row = int(np.flatnonzero(corpus["is_val"] == 1)[0])
context = int(corpus["support_mask"][row].sum())
observations = {
    "spend": corpus["treatment_raw"][row],
    "controls": corpus["covariates"][row],
    "context_outcome": corpus["outcome_raw"][row, :context],
    "channel_mask": corpus["treatment_active_mask"][row],
    "control_mask": corpus["covariate_active_mask"][row],
    "latent_mask": corpus["latent_active_mask"][row],
}
result = decompose(
    trained.model.eval(), observations, n_samples=64, units="sales units", random_seed=7
)
result.components.shape, result.graphs.shape

In [ ]:
result.plot.components()

In [ ]:
result.plot.channels()

In [ ]:
result.plot.channel_totals()

In [ ]:
result.plot.graph()

In [ ]:
result.plot.graph_posterior()

In [ ]:
result.plot.uncertainty()

## Score the draws against the known truth

`result.diagnostics(...)` computes the diagnostic catalog once; its metrics, figures and report all read that one computation. Truth is in the result's component order (baseline, channels, controls, latent total, indirect sources) and natural units; graph truth is the corpus' edge-slot vector.

In [ ]:
truth = np.concatenate(
    [
        corpus["baseline_intrinsic"][row][:, None],
        corpus["treatment_contribution_raw"][row],
        corpus["covariate_contribution"][row],
        corpus["latent_unobserved_contribution"][row].sum(axis=-1, keepdims=True),
        corpus["indirect_effects_by_source"][row],
    ],
    axis=-1,
)[context:].astype(np.float64)
diagnostics = result.diagnostics(
    truth=truth, graph_truth=corpus["g"][row], observed=corpus["outcome_raw"][row, context:]
)
{
    name: diagnostics.metrics[name]
    for name in (
        "attribution/live_media/crps",
        "attribution/dead_media/leakage_scaled_mae",
        "timeavg/channel/cov80",
        "cumulative/channel/energy_score",
        "temporal/channel/lag1_tracking_abs_error",
        "latent_total/r2",
        "closure/draw_relative_error",
        "share/control_share_mae",
    )
}

In [ ]:
# Undefined values keep their reason, for example a family without dead effects.
families = ("metric/attribution/", "metric/timeavg/", "metric/temporal/")
{
    name: status["reason"]
    for name, status in diagnostics.statuses.items()
    if status["status"] == "unavailable" and name.startswith(families)
}

In [ ]:
diagnostics.plot.recovery()

In [ ]:
diagnostics.plot.attribution()

In [ ]:
diagnostics.plot.window_totals()

In [ ]:
diagnostics.plot.cumulative()

In [ ]:
diagnostics.plot.temporal()

In [ ]:
diagnostics.plot.reliability()

In [ ]:
diagnostics.plot.calibration("coverage")

In [ ]:
diagnostics.plot.graph_statistics()

## Write the report

`diagnostics.report(dir)` writes what `build_report` writes (the saved result, the evidence, `diagnostics.json`, one PNG per available panel, `report.html` and a hash-bound `manifest.json`) from the values already computed. `render_report(dir, new_dir)` later rebuilds the figures from those saved files alone, without the model.

In [ ]:
manifest = diagnostics.report(root / "report")
rendered = sorted(
    name for name, panel in manifest["panels"].items() if panel.get("render_status") == "ok"
)
manifest["status"], rendered

Truth-dependent panels need simulation truth, and the live/dead split needs graph truth; without them the panels are unavailable with the reason in `diagnostics.statuses`. Component totals exclude observation noise. One world's PIT/ECDF/coverage figures are dependent screening, not independent-world SBC. Context and query windows are separately sampled marginals.